# getting biomass and carbon for PNG using LCCS BCEs
- load in PNG province files (prepared by Chloe)
- get ESA biomass (through geobox)
- reproject and get Mg correctly for 30m (NOTE: if you compare this to the png_lccs_classification_v0_2_data_merged.tif product it looks misaligned, however it is correct and aligned in the code due to both being load_reproject from GLO30 - basically calculations are correctly align for ESA biomass and LCCS)
- get LCCS for extent (through geobox)
- get BCE from LCCS pixels to sum of biomass from ESA

In [1]:
import sys
import numpy as np
import geopandas as gpd

import fiona
from shapely.geometry import shape

import rioxarray
from rasterio.features import geometry_mask

import datacube
dc = datacube.Datacube(app="extents")
from datacube.utils.aws import configure_s3_access
from dea_tools.datahandling import load_reproject

# 8.4.25 - Matt Paget work around for error CPLE_HttpResponseError: CURL error: Failed to connect to easi-caching-proxy.caching-proxy port 80 after 0 ms: Couldn't connect to server
sys.path.insert(1, "/home/jovyan/code/easi-notebooks/")
from easi_tools.notebook_utils import unset_cachingproxy

In [2]:
# Access AWS "requester-pays" buckets
# This is necessary for reading data from most third-party AWS S3 buckets such as for Landsat and Sentinel-2
configure_s3_access(aws_unsigned=False, requester_pays=True);

In [3]:
# load in PNG province buffered by TSZ (territorial sea zone, 22km buffered to sea to inlcude seagrass)
province_file = '../data/PNG_province_TSZbuffered_EPSG32755/Western_prov_TSZ.shp'
province_df = gpd.read_file(province_file)
province_df

,Id,geometry
0,0,"POLYGON ((-167082.201 9449387.875, -167025.331..."


In [4]:
# This defines the function that converts a linear vector file into a string of x,y coordinates

def geom_query(geom, geom_crs='EPSG:32755'):
    """
    Create datacube query snippet for geometry
    """
    return {
        'x': (geom.bounds[0], geom.bounds[2]),
        'y': (geom.bounds[1], geom.bounds[3]),
        'crs': geom_crs
    }

def warp_geometry(geom, crs_crs, dst_crs):
    """
    warp geometry from crs_crs to dst_crs
    """
    return shapely.geometry.shape(rasterio.warp.transform_geom(crs_crs, dst_crs, shapely.geometry.mapping(geom)))

In [5]:
# use fiona module to open the shape file
province = fiona.open(province_file)

geom_ = shape(province[0]['geometry'])
geom_query_ = geom_query(geom=geom_)

crs = "EPSG:32755"
res = (10, -10)

query =({'output_crs':crs,
         'resolution':res})

query.update(geom_query(geom=geom_, geom_crs=province.crs_wkt))

In [6]:
# load ESA biomass
with unset_cachingproxy():
    ESAbiomass = dc.load(product='cci_biomass_annual_v51', measurements = ['agb', 'agb_sd'], resampling='max', time = ('2020-01-01', '2020-12-31'), **query)

In [7]:
ESAbiomass = ESAbiomass.squeeze('time')

In [8]:
ESAbiomass_Mg10m = (ESAbiomass / 100) # to get Mg at pixel correct pixel size for 10m

In [9]:
# ESAbiomass_Mg10m.odc.write_cog('ESAbiomass_Mg10m.tif', overwrite=True)

In [10]:
# use fiona module to open the shape file
province = fiona.open(province_file)

geom_ = shape(province[0]['geometry'])
geom_query_ = geom_query(geom=geom_)

crs = "EPSG:32755"
res = (30, -30)

query =({'output_crs':crs,
         'resolution':res})

query.update(geom_query(geom=geom_, geom_crs=province.crs_wkt))

In [11]:
# load in GLO data 
with unset_cachingproxy():
    GLO30 = dc.load(product='copernicus_dem_30', resampling='bilinear', time = ('2022-01-01', '2022-12-31'), **query)

In [12]:
# load LCCS band 5 for BCEs
# LCCS data EPSG:32755 30m pixel
LCCS_path = '/home/jovyan/code/livingearth_png/notebooks/png_lccs_classification_v0_2_data_merged.tif'
LCCS_load = load_reproject(path=LCCS_path, how=GLO30.odc.geobox).load()
LCCS_load = LCCS_load.rename('lccs')
LCCS_data = LCCS_load[4]

/env/lib/python3.12/site-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


In [13]:
# LCCS_data
# LCCS_data.odc.write_cog('LCCS_data.tif', overwrite=True)

In [14]:
# Clip the DataArray to the GeoDataFrame
lccs_clipped = LCCS_data.rio.clip([geom_], province_df.crs, drop=False, invert=False)

# Mask out values outside polygon with NaNs (if not already)
lccs_clipped = lccs_clipped.where(~lccs_clipped.isnull(), other=np.nan)

In [15]:
ESAbiomass_reprojected = ESAbiomass_Mg10m.odc.reproject(how=GLO30.odc.geobox, resampling="max")

In [16]:
ESAbiomass_Mg30m = (ESAbiomass_reprojected * 9) # to get Mg at pixel correct pixel size for 30m

In [17]:
# ESAbiomass_Mg30m.agb.odc.write_cog('ESAbiomass_Mg30m.tif', overwrite=True)

In [18]:
mangrove = lccs_clipped == 1
supratidal = lccs_clipped == 2
saltmarsh = lccs_clipped == 3

In [19]:
mangrove_biomass = ESAbiomass_Mg30m.where(mangrove)
supratidal_biomass = ESAbiomass_Mg30m.where(supratidal)
saltmarsh_biomass = ESAbiomass_Mg30m.where(saltmarsh)

In [20]:
# Set numpy print options to suppress scientific notation
np.set_printoptions(suppress=True)

In [21]:
print(mangrove_biomass.sum(skipna=True).agb)
print(mangrove_biomass.sum(skipna=True).agb_sd)

print(supratidal_biomass.sum(skipna=True).agb)
print(supratidal_biomass.sum(skipna=True).agb_sd)

print(saltmarsh_biomass.sum(skipna=True).agb)
print(saltmarsh_biomass.sum(skipna=True).agb_sd)

<xarray.DataArray 'agb' ()> Size: 8B
array(8255619.35999999)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
<xarray.DataArray 'agb_sd' ()> Size: 8B
array(3118731.3)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
<xarray.DataArray 'agb' ()> Size: 8B
array(30574476.80999998)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
<xarray.DataArray 'agb_sd' ()> Size: 8B
array(13145933.34000006)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
<xarray.DataArray 'agb' ()> Size: 8B
array(668111.76)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
<xarray.DataArray 'agb_sd' ()> Size: 8B
array(279159.39)
Coordinates:
    time     datetime64[ns] 8B 2020-01-01
    band     int64 8B 5
